In [3]:
import os
import pandas as pd
import numpy as np
import psutil
import math
from typing import Union

from itertools import product

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from utils.utils import load_data, preprocess_data, measure_duration

# 1.1 Neural Network Implementation 

In [43]:
import numpy as np
import pandas as pd
from typing import Union

class ActivationFunction:
    def __init__(self, function, derivative_function):
        self.function = function
        self.derivative_function = derivative_function

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.function(x)

    def derivative(self, x: np.ndarray) -> np.ndarray:
        return self.derivative_function(x)


class ScratchNeuralNetwork:
    def __init__(
        self,
        activation_function: ActivationFunction,
        num_hidden_layers: int,
        num_nodes_per_hidden_layer: list[int],
        num_nodes_input: int,
        num_nodes_output: int
    ):
        self.activation_function = activation_function

        if num_hidden_layers != len(num_nodes_per_hidden_layer):
            raise ValueError("Number of hidden layers must match the size of the nodes list.")

        self.weight_matrix_list = []
        self.bias_vector_list = []
        self.input_node_count = num_nodes_input

        prev_layer_node_count = num_nodes_input

        for hidden_layer_node_count in num_nodes_per_hidden_layer:
            self.weight_matrix_list.append(
                np.random.randn(hidden_layer_node_count, prev_layer_node_count) * np.sqrt(1 / prev_layer_node_count)
            )
            self.bias_vector_list.append(np.zeros((hidden_layer_node_count, 1)))
            prev_layer_node_count = hidden_layer_node_count

        self.weight_matrix_list.append(
            np.random.randn(num_nodes_output, prev_layer_node_count) * np.sqrt(1 / prev_layer_node_count)
        )
        self.bias_vector_list.append(np.zeros((num_nodes_output, 1)))

    def count_parameters(self) -> int:
        total_params = 0
        for weights, biases in zip(self.weight_matrix_list, self.bias_vector_list):
            total_params += weights.size + biases.size
        return total_params

    def softmax_function(self, x: np.ndarray) -> np.ndarray:
        x = np.clip(x, -700, 700)  # Prevent overflow in exp
        x = x - np.max(x, axis=0, keepdims=True)  # stability trick
        exp_x = np.exp(x)
        sum_exp = np.sum(exp_x, axis=0, keepdims=True)
        return exp_x / (sum_exp + 1e-15)

    def feed_forward(self, input_data_point: np.ndarray):
        a = input_data_point.reshape(-1, 1)
        activations = [a]
        zs = []

        for W, b in zip(self.weight_matrix_list[:-1], self.bias_vector_list[:-1]):
            z = np.dot(W, a) + b
            a = self.activation_function(z)
            zs.append(z)
            activations.append(a)

        z = np.dot(self.weight_matrix_list[-1], a) + self.bias_vector_list[-1]
        a = self.softmax_function(z)
        zs.append(z)
        activations.append(a)

        return activations, zs

    def back_propagate(self, input_data_point: np.ndarray, true_label_vector: np.ndarray, learning_rate: float):
        activations, zs = self.feed_forward(input_data_point)

        y = true_label_vector.reshape(-1, 1)
        delta = activations[-1] - y

        for l in reversed(range(len(self.weight_matrix_list))):
            a_prev = activations[l]
            dW = np.dot(delta, a_prev.T)
            db = delta

            self.weight_matrix_list[l] -= learning_rate * dW
            self.bias_vector_list[l] -= learning_rate * db

            if l != 0:
                delta = np.dot(self.weight_matrix_list[l].T, delta) * self.activation_function.derivative(activations[l])

    def predict(self, input_data: Union[np.ndarray, pd.DataFrame]) -> np.ndarray:
        if isinstance(input_data, pd.DataFrame):
            input_array = input_data.values
        elif isinstance(input_data, np.ndarray):
            input_array = input_data
        else:
            raise TypeError("Input must be a NumPy array or pandas DataFrame.")

        if input_array.ndim == 1:
            input_array = input_array.reshape(1, -1)

        if input_array.shape[1] != self.input_node_count:
            raise ValueError(f"Each input vector must have {self.input_node_count} features.")

        results = []
        for data_point in input_array:
            activations, _ = self.feed_forward(data_point)
            results.append(activations[-1].flatten())

        return np.vstack(results)

    def train(
        self,
        X_train: Union[np.ndarray, list],
        y_train: Union[np.ndarray, list],
        epochs: int = 10,
        learning_rate: float = 0.001,
        batch_size: int = 30
    ):
        X_train = np.array(X_train)
        y_train = np.array(y_train)

        if X_train.ndim != 2:
            raise ValueError(f"X_train must be 2D, got {X_train.shape}")
        if y_train.ndim != 2:
            raise ValueError("y_train must be one-hot encoded and 2D")
        if X_train.shape[0] != y_train.shape[0]:
            raise ValueError("Mismatched number of samples in X_train and y_train")

        num_samples = X_train.shape[0]

        for epoch in range(epochs):
            indices = np.arange(num_samples)
            np.random.shuffle(indices)
            X_train = X_train[indices]
            y_train = y_train[indices]

            epoch_loss = 0.0
            for start_idx in range(0, num_samples, batch_size):
                end_idx = min(start_idx + batch_size, num_samples)
                batch_X = X_train[start_idx:end_idx]
                batch_y = y_train[start_idx:end_idx]

                batch_loss = 0.0
                for x, y in zip(batch_X, batch_y):
                    output, _ = self.feed_forward(x)
                    loss_val = -np.sum(y.reshape(-1, 1) * np.log(output[-1] + 1e-15))
                    batch_loss += loss_val
                    self.back_propagate(x, y, learning_rate)

                epoch_loss += batch_loss / len(batch_X)

            epoch_loss /= (num_samples / batch_size)
            print(f"Epoch {epoch + 1}/{epochs} - Loss: {epoch_loss:.6f}")
    
    def compute_ram_usage(self):
        process = psutil.Process(os.getpid())
        mem_bytes = process.memory_info().rss  # Resident Set Size
        return mem_bytes / (1024 ** 2)


Sigmoid = ActivationFunction(
    function=lambda x: 1 / (1 + np.exp(-x)),
    derivative_function=lambda a: a * (1 - a)
)

ReLU = ActivationFunction(
    function=lambda x: np.maximum(0, x),
    derivative_function=lambda a: np.where(a > 0, 1.0, 0.0)
)

tanh = ActivationFunction(
    function=lambda x: np.tanh(x),
    derivative_function=lambda a: 1 - a**2
)

# 1.2 Dataset Loading 

In [5]:
# Load the datasets
df_maternal = load_data('../data/raw/uci/maternal_health_risk/maternal_health_risk.csv')
df_amazon = load_data('../data/raw/kaggle/reviews/amazon_review_ID.shuf.lrn.csv')

Data successfully loaded from ../data/raw/uci/maternal_health_risk/maternal_health_risk.csv. First 5 rows:
   Age  SystolicBP  DiastolicBP    BS  BodyTemp  HeartRate  RiskLevel
0   25         130           80  15.0      98.0         86  high risk
1   35         140           90  13.0      98.0         70  high risk
2   29          90           70   8.0     100.0         80  high risk
3   30         140           85   7.0      98.0         70  high risk
4   35         120           60   6.1      98.0         76   low risk
Data successfully loaded from ../data/raw/kaggle/reviews/amazon_review_ID.shuf.lrn.csv. First 5 rows:
   ID  V1  V2  V3  V4  V5  V6  V7  V8  V9  ...  V9992  V9993  V9994  V9995  \
0   0  17   4   8   8   9   4   0   2   3  ...      0      0      0      0   
1   1  21   9   5   8   6   2  16   3  12  ...      0      0      0      2   
2   2   9   7   6   3   8   2   9   4   4  ...      0      0      0      0   
3   3   8   3   5   2   4   3   8   2   4  ...      0      

In [6]:
# Example for Dataset 1:
if df_maternal is not None:
    X1, y1 = preprocess_data(df_maternal, label_col="RiskLevel")

# Example for Dataset 2:
if df_amazon is not None:
    X2, y2 = preprocess_data(df_amazon, label_col="Class")

Preprocessing complete: 1014 samples, 6 features, 3 classes.
Preprocessing complete: 750 samples, 10001 features, 50 classes.


In [7]:
# 1) Train/test split for dataset 1
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)

# 2) One-hot encoding for labels of dataset 1
ohe1 = OneHotEncoder()
y1_train_ohe = ohe1.fit_transform(y1_train.reshape(-1, 1)).toarray()
y1_test_ohe  = ohe1.transform(y1_test.reshape(-1, 1)).toarray()

# Output the shapes of the datasets
print("Dataset 1 shapes:")
print(f"  X1_train: {X1_train.shape}, y1_train_ohe: {y1_train_ohe.shape}")
print(f"  X1_test : {X1_test.shape},  y1_test_ohe : {y1_test_ohe.shape}")

# 3) Train/test split for dataset 2
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

# 4) One-hot encoding for labels of dataset 2
ohe2 = OneHotEncoder()
y2_train_ohe = ohe2.fit_transform(y2_train.reshape(-1, 1)).toarray()
y2_test_ohe  = ohe2.transform(y2_test.reshape(-1, 1)).toarray()

# Output the shapes of the datasets
print("\nDataset 2 shapes:")
print(f"  X2_train: {X2_train.shape}, y2_train_ohe: {y2_train_ohe.shape}")
print(f"  X2_test : {X2_test.shape},  y2_test_ohe : {y2_test_ohe.shape}")

Dataset 1 shapes:
  X1_train: (811, 6), y1_train_ohe: (811, 3)
  X1_test : (203, 6),  y1_test_ohe : (203, 3)

Dataset 2 shapes:
  X2_train: (600, 10001), y2_train_ohe: (600, 50)
  X2_test : (150, 10001),  y2_test_ohe : (150, 50)


In [50]:
from sklearn.metrics import log_loss


activations_list = [Sigmoid, ReLU, tanh]
hidden_units_list = [8, 16, 32]

def run_experiments(X_train, y_train_ohe, X_test, y_test_ohe,
                    input_size, output_size, epochs=500, learning_rate=0.01, verbose=False):
    results = []
    y_test_labels = y_test_ohe.argmax(axis=1)
    for activation in activations_list:
        for hu in hidden_units_list:
            nn = ScratchNeuralNetwork(num_nodes_input= input_size, num_hidden_layers=1, num_nodes_per_hidden_layer=[hu],num_nodes_output=output_size, activation_function=activation)
            nn.train(X_train, y_train_ohe, epochs, learning_rate)
            preds = nn.predict(X_test)
            predicted_labels = preds.argmax(axis=1)
            acc = accuracy_score(y_test_labels, predicted_labels)
            f1 = f1_score(y_test_labels, predicted_labels, average='weighted')
            loss = log_loss(y_test_ohe, preds)
            results.append({
                'Activation': activation,
                'Hidden Layers': 1,
                'Hidden Units': hu,
                'Accuracy': f"{acc*100:.2f}%",
                'F1 Score': f"{f1:.4f}",
                'Final Loss': f"{loss:.4f}",
                'Parameters': nn.count_parameters(),
                'RAM Used (MB)': f"{nn.compute_ram_usage():.2f}"
            })
    return pd.DataFrame(results)

In [51]:
def grid_search(GRID, X_train, y_train_ohe, X_test, y_test_ohe, input_size, output_size, epochs=500, verbose=False):
    best, hist = None, []
    y_true = y_test_ohe.argmax(axis=1)
    for hl, hu, act, lr in product(GRID["hidden_layers"], GRID["hidden_units"], GRID["activation"], GRID["lr"]):
        nn = ScratchNeuralNetwork(num_nodes_input= input_size, num_hidden_layers=hl, num_nodes_per_hidden_layer=[hu], num_nodes_output=output_size, activation_function=act)
        nn.train(X_train, y_train_ohe, epochs, lr, 30)
        preds = nn.predict(X_test)
        acc  = accuracy_score(y_true, preds)
        f1 = f1_score(y_true, preds, average='weighted')
        loss = ((y_test_ohe - nn.forward(X_test))**2).mean()
        row  = {
            "Activation": act.capitalize(),
            "Hidden Layers": hl,
            "Hidden Units": hu,
            "Learning Rate": lr,
            "Accuracy": acc,
            "F1 Score": f1,
            "Final Loss": loss,
            "Parameters": nn.count_parameters(),
            "RAM MB": nn.compute_ram_usage()
        }
        hist.append(row)
        if best is None or acc > best["Accuracy"]:
            best = row
    return pd.DataFrame(hist), best

In [52]:
df1_results, s1, e1, dur1 = measure_duration(
    run_experiments,
    X_train=X1_train, y_train_ohe=y1_train_ohe,
    X_test=X1_test,  y_test_ohe=y1_test_ohe,
    input_size=X1_train.shape[1], output_size=y1_train_ohe.shape[1],
    epochs=500, learning_rate=0.01, verbose=False
)


print(f"Implemntation Dataset 1 took {dur1}")

Epoch 1/500 - Loss: 1.046399
Epoch 2/500 - Loss: 0.936689
Epoch 3/500 - Loss: 0.872145
Epoch 4/500 - Loss: 0.836197
Epoch 5/500 - Loss: 0.809011
Epoch 6/500 - Loss: 0.817092
Epoch 7/500 - Loss: 0.802271
Epoch 8/500 - Loss: 0.792184
Epoch 9/500 - Loss: 0.798718
Epoch 10/500 - Loss: 0.822166
Epoch 11/500 - Loss: 0.767833
Epoch 12/500 - Loss: 0.781245
Epoch 13/500 - Loss: 0.794117
Epoch 14/500 - Loss: 0.798114
Epoch 15/500 - Loss: 0.786760
Epoch 16/500 - Loss: 0.777653
Epoch 17/500 - Loss: 0.774875
Epoch 18/500 - Loss: 0.778806
Epoch 19/500 - Loss: 0.755864
Epoch 20/500 - Loss: 0.816803
Epoch 21/500 - Loss: 0.757078
Epoch 22/500 - Loss: 0.816207
Epoch 23/500 - Loss: 0.829246
Epoch 24/500 - Loss: 0.751342
Epoch 25/500 - Loss: 0.771708
Epoch 26/500 - Loss: 0.752928
Epoch 27/500 - Loss: 0.747461
Epoch 28/500 - Loss: 0.749609
Epoch 29/500 - Loss: 0.741341
Epoch 30/500 - Loss: 0.741003
Epoch 31/500 - Loss: 0.765948
Epoch 32/500 - Loss: 0.766094
Epoch 33/500 - Loss: 0.778563
Epoch 34/500 - Loss

In [53]:
print(df1_results)
# display(df1_results)

                                          Activation  Hidden Layers  \
0  <__main__.ActivationFunction object at 0x12645...              1   
1  <__main__.ActivationFunction object at 0x12645...              1   
2  <__main__.ActivationFunction object at 0x12645...              1   
3  <__main__.ActivationFunction object at 0x12645...              1   
4  <__main__.ActivationFunction object at 0x12645...              1   
5  <__main__.ActivationFunction object at 0x12645...              1   
6  <__main__.ActivationFunction object at 0x12644...              1   
7  <__main__.ActivationFunction object at 0x12644...              1   
8  <__main__.ActivationFunction object at 0x12644...              1   

   Hidden Units Accuracy F1 Score Final Loss  Parameters RAM Used (MB)  
0             8   62.56%   0.6095     0.7208          83         48.47  
1            16   67.49%   0.6709     0.6494         163         39.81  
2            32   67.49%   0.6742     0.7056         323         39.72

In [ ]:
df2_results, s3, e3, dur2 = measure_duration(
    run_experiments,
    X_train=X2_train, y_train_ohe=y2_train_ohe,
    X_test=X2_test,  y_test_ohe=y2_test_ohe,
    input_size=X2_train.shape[1], output_size=y2_train_ohe.shape[1],
    hahahahha=500, learning_rate=0.01, verbose=True
)

print(f"Implemntation Dataset 2 took {dur2}")

Epoch 1/500 - Loss: 3.729897
Epoch 2/500 - Loss: 3.085738
Epoch 3/500 - Loss: 2.873664
Epoch 4/500 - Loss: 2.714976
Epoch 5/500 - Loss: 2.576270
Epoch 6/500 - Loss: 2.445994
Epoch 7/500 - Loss: 2.323095
Epoch 8/500 - Loss: 2.209035
Epoch 9/500 - Loss: 2.102401
Epoch 10/500 - Loss: 2.004099
Epoch 11/500 - Loss: 1.911694
Epoch 12/500 - Loss: 1.825018
Epoch 13/500 - Loss: 1.743757
Epoch 14/500 - Loss: 1.668741
Epoch 15/500 - Loss: 1.598638
Epoch 16/500 - Loss: 1.533232
Epoch 17/500 - Loss: 1.471480
Epoch 18/500 - Loss: 1.413559
Epoch 19/500 - Loss: 1.359524
Epoch 20/500 - Loss: 1.308373
Epoch 21/500 - Loss: 1.260825
Epoch 22/500 - Loss: 1.216026
Epoch 23/500 - Loss: 1.173760
Epoch 24/500 - Loss: 1.133047
Epoch 25/500 - Loss: 1.095473
Epoch 26/500 - Loss: 1.059834
Epoch 27/500 - Loss: 1.026364
Epoch 28/500 - Loss: 0.994033
Epoch 29/500 - Loss: 0.963486
Epoch 30/500 - Loss: 0.934028
Epoch 31/500 - Loss: 0.907026
Epoch 32/500 - Loss: 0.881371
Epoch 33/500 - Loss: 0.857251
Epoch 34/500 - Loss

ValueError: Input contains NaN.

In [56]:
print(df2_results)
# display(df2_results)

NameError: name 'df2_results' is not defined

In [ ]:
grid_1 = {
    "hidden_layers": [1, 2, 3, 4, 5],
    "hidden_units":  [2, 4, 8, 16, 32, 64],
    "activation":    [Sigmoid, ReLU, tanh],
    "lr":            [0.001, 0.05, 0.01]
}

(hist1, best1), s1, e1, dur3 = measure_duration(
    grid_search,
    GRID=grid_1,
    X_train=X1_train, y_train_ohe=y1_train_ohe,
    X_test=X1_test, y_test_ohe=y1_test_ohe,
    input_size=X1_train.shape[1],
    output_size=y1_train_ohe.shape[1],
    epochs=1000, verbose=False
)

print(f"Grid Search (Dataset 1) took {dur3}")


In [ ]:
grid_1 = {
    "hidden_layers": [1, 2, 3, 4, 5],
    "hidden_units":  [2, 4, 8, 16, 32, 64],
    "activation":    ["sigmoid", "relu", "tanh"],
    "lr":            [0.1, 0.05, 0.01]
}

(hist1, best1), s1, e1, dur3 = measure_duration(
    grid_search,
    GRID=grid_1,
    X_train=X1_train, y_train_ohe=y1_train_ohe,
    X_test=X1_test, y_test_ohe=y1_test_ohe,
    input_size=X1_train.shape[1],
    output_size=y1_train_ohe.shape[1],
    epochs=1000, verbose=False
)

print(f"Grid Search (Dataset 1) took {dur3}")


In [ ]:
best1_df = pd.DataFrame([best1])
hist1_df = pd.DataFrame(hist1)

In [ ]:
print(best1_df)
# display(hist1)

In [ ]:
print(hist1_df)

In [ ]:
grid_2 = {
    "hidden_layers": [1, 2, 3],
    "hidden_units":  [8, 16, 32, 64],
    "activation":    ["sigmoid", "relu", "tanh"],
    "lr":            [0.1, 0.05, 0.01]
}

(hist2, best2), s4, e4, dur4 = measure_duration(
    grid_search,
    GRID=grid_2, 
    X_train=X2_train, y_train_ohe=y2_train_ohe,
    X_test=X2_test, y_test_ohe=y2_test_ohe,
    input_size=X2_train.shape[1],
    output_size=y2_train_ohe.shape[1],
    epochs=1000, verbose=False
)

print(f"Grid Search (Dataset 2) took {dur4}")


In [ ]:
best2_df = pd.DataFrame([best2])
hist2_df = pd.DataFrame(hist2)

In [ ]:
print(best2_df)
# display(hist1)

In [ ]:
print(hist2_df)